In [ ]:
import psycopg2
import os
import pandas as pd
import yaml

In [ ]:
os.environ["PGSERVICEFILE"] = os.path.join(os.getcwd(), ".pg_service.conf")
os.environ["PGPASSFILE"] = os.path.join(os.getcwd(), ".pgpass")

In [ ]:
beaver = psycopg2.connect(service="beam_neb0",password=pgpass)
cursor = beaver.cursor()

In [ ]:
## schema
cursor.execute("""
    select
        table_schema,
        table_name,
        column_name,
        data_type,
        is_nullable,
        column_default
    from information_schema.columns
    where table_schema = 'public'
    order by table_schema, table_name, ordinal_position;
""")

public_schema = pd.DataFrame(cursor.fetchall(), columns=[
    "schema", "table", "column", "data_type", "nullable", "default"
])

In [ ]:
output_dir = os.path.join(os.getcwd(), "SL")

for table, group in columns_df.groupby("table"):
    table_def = {
        "version": 2,
        "models": [{
            "name": table,
            "description": "",
            "columns": [
                {
                    "name": row["column"],
                    "description": "",
                    "data_type": row["data_type"],
                    "nullable": row["nullable"] == "YES"
                }
                for _, row in group.iterrows()
            ]
        }]
    }
    
    filepath = os.path.join(output_dir, f"{table}.yml")
    with open(filepath, "w") as f:
        yaml.dump(table_def, f, default_flow_style=False, sort_keys=False)

print(f"Generated {len(columns_df['table'].unique())} YML files in ./{output_dir}/")